# Self-Improvement & MLflow, Part 3: Automate via CI/CD

The code from [the article](https://opendatascience.com/self-improvement-mlflow-part-3/), in article order. Each code cell is also saved as a file in this folder.

The snippets show the MLflow calls, not a complete agent: names such as `agent` or `run_agent`, and the helpers marked *our helper*, are yours to supply. Each cell says which names it expects. A snippet that continues an earlier one doesn't repeat its imports.

## 1. Track the Baseline

File: `baseline.py`

Not defined here: `get_dataset` is from `mlflow.genai.datasets`; `load_checkpoint` is in `checkpoints.py`; `tests_per_theme`, `human_edits_per_test` and `traces_per_theme` are your own helpers.

In [ ]:
import mlflow

dataset = get_dataset(name="support_regressions")
last_run_ms = load_checkpoint("triage")                  # our helper, defined in Section 2
new_traces = mlflow.search_traces(
    filter_string=f"attributes.timestamp_ms > {last_run_ms}",
    return_type="list")

with mlflow.start_run(run_name="loop-baseline"):
    mlflow.log_metrics({
        "tests_per_theme": tests_per_theme(dataset),           # our helper
        "human_edits_per_test": human_edits_per_test(dataset), # our helper
        "traces_per_theme": traces_per_theme(new_traces),      # our helper
        "incoming_traces": len(new_traces),
    })


## 2. Schedule the Trace Ingestion

File: `loop/ingest.py`

Not defined here: `load_checkpoint` and `save_checkpoint` are in `checkpoints.py`; `TRIAGE_THRESHOLD` and `run_triage_agent` are your own.

In [ ]:
last_run_ms = load_checkpoint("triage")   # checkpoint helper
new_failures = mlflow.search_traces(
    filter_string=(f"feedback.quality = 'Fail' "
                   f"AND expectation.expected_response IS NOT NULL "
                   f"AND attributes.timestamp_ms > {last_run_ms}"),
    return_type="list")

if len(new_failures) >= TRIAGE_THRESHOLD:
    dataset = get_dataset(name="support_regressions")
    dataset = dataset.merge_records(new_failures)
    save_checkpoint("triage", int(time.time() * 1000))   # our helper
    run_triage_agent(dataset_version=dataset.version, traces=new_failures)


## 2. Schedule the Trace Ingestion: checkpoints

File: `checkpoints.py`

Not defined here: `EXPERIMENT_ID` is your experiment's id.

In [ ]:
from mlflow import MlflowClient

client = MlflowClient()

def load_checkpoint(name, experiment_id=EXPERIMENT_ID):
    tags = client.get_experiment(experiment_id).tags
    return int(tags.get(f"checkpoint.{name}", 0))

def save_checkpoint(name, timestamp_ms, experiment_id=EXPERIMENT_ID):
    client.set_experiment_tag(experiment_id, f"checkpoint.{name}", str(timestamp_ms))


## 2. Schedule the Trace Ingestion: tag the dataset

File: `dataset_tags.py`

Not defined here: `dataset`, `records` and `triage_run_id` come from the triage run.

In [ ]:
dataset = dataset.merge_records(records)
dataset = dataset.set_tags({
    "generated_by": "triage-agent",
    "source_run": triage_run_id,
    "tests_per_theme_cap": "3",
})


## 3. Amplify the Signal

File: `grounded_in_policy.py`

In [ ]:
from mlflow.entities import Feedback, SpanStatusCode, SpanType
from mlflow.genai.judges import make_judge
from mlflow.genai.scorers import scorer, ScorerSamplingConfig

# export TYPESAFE_API_KEY=...
policy_judge = make_judge(
    name="grounded_in_policy",
    instructions=(
        "Answer yes or no. Does {{ outputs }} make only claims that the "
        "policy text in {{ inputs }} supports?"
    ),
    model="typesafe:/jev-latest",
    feedback_value_type=bool,
)

@scorer
def grounded_in_policy(trace):
    lookups = trace.search_spans(span_type=SpanType.TOOL, name="lookup_policy")
    if not lookups:
        return Feedback(value=False,
                        rationale="The agent answered without calling lookup_policy.")

    call = lookups[-1]
    if call.status.status_code == SpanStatusCode.ERROR:
        return Feedback(value=False,
                        rationale="lookup_policy failed, so the answer has no policy behind it.")
    if not (call.inputs or {}).get("topic") or not call.outputs:
        return Feedback(value=False,
                        rationale="lookup_policy was called without a topic or returned nothing.")

    return policy_judge(
        inputs={"question": trace.data.request, "policy": call.outputs},
        outputs=trace.data.response,
    )

registered = grounded_in_policy.register(name="grounded_in_policy")
registered.start(
    sampling_config=ScorerSamplingConfig(
        sample_rate=0.1,
        filter_string="tags.environment = 'production'",
    )
)

# Judge verdicts carry metadata["typesafe.probability"]


## 5. Automate the Alignment

File: `loop/align_judge.py`

Not defined here: `quality_judge`, `experiment_id`, `AGREEMENT_THRESHOLD`, `split_traces` and `agreement_with_humans` are your own.

In [ ]:
from mlflow.genai.judges.optimizers import MemAlignOptimizer

paired = mlflow.search_traces(filter_string="feedback.quality IS NOT NULL",
                              return_type="list")
fit, check = split_traces(paired, check_fraction=0.3)   # our helper

before = agreement_with_humans(quality_judge, check)    # our helper
if len(paired) >= 10 and before < AGREEMENT_THRESHOLD:
    aligned = quality_judge.align(
        fit, MemAlignOptimizer(reflection_lm="openai:/gpt-5-mini"))
    if agreement_with_humans(aligned, check) > before:
        aligned.register(experiment_id=experiment_id)
# Can assign an alias here if you're confident in the loop


## 6. Automate the Approval

File: `tests/regression/test_support_agent.py`

Not defined here: `get_dataset` is from `mlflow.genai.datasets`; `run_agent` is your agent; `grounded_in_policy` comes from `grounded_in_policy.py`.

In [ ]:
# tests/regression/test_support_agent.py
import mlflow
from mlflow.genai.scorers import Correctness

@mlflow.test
def test_support_regressions():
    dataset = get_dataset(name="support_regressions")
    result = mlflow.genai.evaluate(
        data=dataset,
        predict_fn=run_agent,
        scorers=[grounded_in_policy, Correctness()],
    )
    assert result.passed, result.reason


## 6. Automate the Approval: approve proposed tests

File: `loop/approve_tests.py`

Not defined here: `proposed_tests`, `dataset`, `CRITERIA`, `TESTS_PER_THEME_CAP` and the check helpers are your own.

In [ ]:
def approve(test):
    return all([
        fails_on_source_trace(test),                      # our helper: reproduces the failure
        not_near_duplicate(test, dataset),                # our helper
        count_in_theme(dataset, test.theme) < TESTS_PER_THEME_CAP,   # our helper
        reviewer_agent_approves(test, CRITERIA),          # our helper: not the proposing agent
    ])

approved = [t for t in proposed_tests if approve(t)]
flagged = [t for t in proposed_tests if t not in approved]

dataset = dataset.merge_records(approved)
dataset = dataset.set_tags({"holdout_fraction": "0.2"})
send_to_human_queue(flagged)                              # our helper


## 7. Improve Other Components: judges for a skill

File: `skill_judges.py`

In [ ]:
from typing import Literal
from mlflow import MlflowClient
from mlflow.entities import Feedback
from mlflow.genai.judges import make_judge
from mlflow.genai.scorers import scorer

skill_loaded = make_judge(
    name="tracing-skill-invoked",
    instructions=(
        "Answer yes or no. Does {{ trace }} show the agent loading the "
        "instrumenting-with-mlflow-tracing skill before it edited any code?"
    ),
    model="openai:/gpt-4o-mini",
    feedback_value_type=Literal["yes", "no"],
)

@scorer
def dataset_created(trace):
    eval_experiment_id = trace.info.tags["eval_experiment_id"]   # set by run_claude, one fresh experiment per run
    found = MlflowClient().search_datasets(experiment_ids=[eval_experiment_id])
    return Feedback(
        value=bool(found),
        rationale="An evaluation dataset exists." if found
        else "The agent never created an evaluation dataset.",
    )


## 7. Improve Other Components: the improvement loop

File: `loop/improve_skill.py`

Not defined here: `skill_loaded` and `dataset_created` come from `skill_judges.py`; `SKILL_DIR`, `MAX_ROUNDS`, `SKILL_CONFIGS`, `CLAUDE_EXPERIMENT_ID` and the helpers are your own.

In [ ]:
candidate = copy_skill(SKILL_DIR)                           # our helper

for _ in range(MAX_ROUNDS):
    round_start = int(time.time() * 1000)
    for config in SKILL_CONFIGS:                            # our helper: one prompt and project per case
        run_claude(config, skills=candidate)                # our helper: runs claude -p with tracing on, in a fresh evaluation experiment tagged on the trace

    traces = mlflow.search_traces(
        experiment_ids=[CLAUDE_EXPERIMENT_ID],
        filter_string=f"attributes.timestamp_ms > {round_start}",
        return_type="list")
    result = mlflow.genai.evaluate(
        data=traces, scorers=[skill_loaded, dataset_created])

    if result.passed:
        open_pull_request(candidate)                        # our helper
        break
    ask_claude(f"These judges failed: {result.reason}. Fix SKILL.md.",
               cwd=candidate)                               # our helper


## 8. Run the Loop in CI

File: `self-improvement.yml`

```yaml
# .github/workflows/self-improvement.yml
name: self-improvement
on:
  schedule:
    - cron: "0 6 * * *"
  pull_request:

env:
  MLFLOW_TRACKING_URI: ${{ secrets.MLFLOW_TRACKING_URI }}
  OPENAI_API_KEY: ${{ secrets.OPENAI_API_KEY }}

jobs:
  ingest:                                   # Section 2
    if: github.event_name == 'schedule'
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - run: pip install "mlflow[genai]"
      - run: mlflow agent setup                 # connect the coding agent to MLflow
      - run: |
          mlflow traces search --experiment-id 1 \
            --filter-string "feedback.quality = 'Fail'" \
            --output json > /tmp/failing_examples.json
      - run: python loop/ingest.py

  align-judge:                              # Section 5
    if: github.event_name == 'schedule'
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - run: pip install "mlflow[genai]" dspy
      - run: python loop/align_judge.py

  verify:                                   # Section 6
    needs: [ingest, align-judge]
    if: ${{ !failure() && !cancelled() }}
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - run: pip install "mlflow[genai]" pytest pytest-xdist
      - run: python loop/approve_tests.py
      - run: pytest -n auto tests/regression

  improve:                                  # Section 7
    needs: verify
    if: github.event_name == 'schedule'
    runs-on: ubuntu-latest
    permissions:
      contents: write
      pull-requests: write
    steps:
      - uses: actions/checkout@v4
      - run: pip install "mlflow[genai]"
      - run: python loop/improve_skill.py
        env:
          GH_TOKEN: ${{ secrets.GITHUB_TOKEN }}
```